Salesperson Analysis
Ranks salespeople by sales, profit, and volume contribution, with precomputed percentile ranks for instant threshold filtering.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.salesperson_analysis import build_salesperson_summary, filter_by_percentile
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])
print(f"Analysis silver: {analysis_silver.shape}")

Build salesperson summary

In [0]:
salesperson_summary = build_salesperson_summary(analysis_silver)
print(salesperson_summary.shape)
print(salesperson_summary[["salesperson_name", "total_sales", "sales_percentile", "total_profit", "unique_customers"]].head(15))

Save

In [0]:
save_gold(blob_service, salesperson_summary,f"{ANALYSIS_BASE}/salesperson_ranking.parquet")

buffer = io.BytesIO()
salesperson_summary.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/salesperson_ranking.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved salesperson_ranking.parquet and .xlsx")

Demo: threshold as filter, plus active/lapsed split

In [0]:
dbutils.widgets.text("sales_percentile_threshold", "90")
sales_threshold = float(dbutils.widgets.get("sales_percentile_threshold"))

top_performers = filter_by_percentile(salesperson_summary, "sales", sales_threshold)
print(f"Top performers (>= {sales_threshold}th percentile): {len(top_performers)}")
print(top_performers[["salesperson_name", "total_sales", "total_profit", "unique_customers", "is_active"]])

lapsed_top = top_performers[~top_performers["is_active"]]
print(f"\nTop performers who've gone quiet (>90 days): {len(lapsed_top)}")
print(lapsed_top[["salesperson_name", "total_sales", "last_sale", "days_since_last_sale"]])